In [4]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np


In [5]:
%pip install db-dtypes


Note: you may need to restart the kernel to use updated packages.


In [6]:
# путь к ключу доступа к Google Cloud (скачанный JSON файл)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credentials.json"

# 2. Инициализируем клиента
client = bigquery.Client()

In [7]:

# финальный SQL запрос. Уровень транзакций. Смотрим только тех кто принял участие в тесте и купил. 
# (скопирован из бигквери в запросах ab_rozetka_aggdata_crm_ga4)
sql_query = """
-- Список всех уникальных участников теста и их групп
WITH test_users AS (
  SELECT DISTINCT
    user_pseudo_id,
    (SELECT value.string_value FROM UNNEST(user_properties) WHERE key = 'ab_test_rozetka_variant') AS user_group
  FROM `ga4-vitberry.analytics_450794542.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20260401' AND '20260407'
    AND (SELECT value.string_value FROM UNNEST(user_properties) WHERE key = 'ab_test_rozetka_variant') IS NOT NULL
),

-- Время самого первого взаиможействия только для участников теста
user_first_sessions AS (
  SELECT 
    user_pseudo_id,
    MIN(event_timestamp) as session_start_raw
  FROM `ga4-vitberry.analytics_450794542.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20260401' AND '20260407'
    AND event_name = 'session_start'
    AND user_pseudo_id IN (SELECT user_pseudo_id FROM test_users) -- Фильтр по тестовім юзерам
  GROUP BY user_pseudo_id
),

-- Данные по покупкам только для участников теста
user_purchases AS (
  SELECT DISTINCT
    user_pseudo_id,
    ecommerce.transaction_id,
    device.category,
    (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_number') AS session_number,
    geo.city,
    CONCAT(IFNULL(session_traffic_source_last_click.manual_campaign.source, 'direct'), '/', IFNULL(session_traffic_source_last_click.manual_campaign.medium, 'none')) AS source_medium,
    event_timestamp as purchase_time_raw
  FROM `ga4-vitberry.analytics_450794542.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20260401' AND '20260407'
    AND event_name = 'purchase'
    AND user_pseudo_id IN (SELECT user_pseudo_id FROM test_users) -- Фильтр по тестовім юзерам
)

-- Склеиваю всё в одну таблицу
SELECT 
  u.user_group,
  TIMESTAMP_MICROS(s.session_start_raw) as session_start,
  TIMESTAMP_MICROS(p.purchase_time_raw) as purchase_time,
  p.* EXCEPT(user_pseudo_id, purchase_time_raw), -- за исключением ID и времени, так как уже есть в селекте
  c.* EXCEPT(site_order_id), -- исключаю так как есть значение transaction_id из таблицы покупок
  u.user_pseudo_id
FROM test_users u
LEFT JOIN user_first_sessions s ON u.user_pseudo_id = s.user_pseudo_id
LEFT JOIN user_purchases p ON u.user_pseudo_id = p.user_pseudo_id
LEFT JOIN `ga4-vitberry.raw_keycrm.view_ab_rozetka_orders` c ON p.transaction_id = c.site_order_id
"""

# перобразую результат запроса в датафрейм для анализа
df = client.query(sql_query).to_dataframe()



e:\Anton\da\miniconda3_marusya_user\envs\vit_analytics_env\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
# Уровень поведения. Смотрю всех кто участвовал в тесте, независимо от того, купил он или нет. 
# (скопирован из бигквери в запросах - ab_rozetka_raw_funnel)

sql_query_behavior = """
 WITH user_list AS (
  SELECT DISTINCT
    user_pseudo_id,
    (SELECT value.string_value FROM UNNEST(user_properties) WHERE key = 'ab_test_rozetka_variant') AS user_group,
  FROM `ga4-vitberry.analytics_450794542.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20260401' AND '20260407'
    AND (SELECT value.string_value FROM UNNEST(user_properties) WHERE key = 'ab_test_rozetka_variant') IS NOT NULL
),

user_behavior AS (
  -- Здесь каждая строка — это ОДИН пользователь
  SELECT
    user_pseudo_id,
    COUNTIF(event_name = 'view_item') AS user_item_views,
    COUNTIF(event_name = 'add_to_cart') AS user_carts,
    -- Считаем транзакции ТОЛЬКО если это событие покупки
    COUNT(DISTINCT IF(event_name = 'purchase', ecommerce.transaction_id, NULL)) AS user_orders,
    -- Суммируем доход ТОЛЬКО из событий покупки
    SUM(IF(event_name = 'purchase', IFNULL(ecommerce.purchase_revenue, 0), 0)) AS user_revenue,
  FROM `ga4-vitberry.analytics_450794542.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20260401' AND '20260407'
  GROUP BY 1
)

SELECT
  u.user_group,
  b.*
FROM user_list u
LEFT JOIN user_behavior b ON u.user_pseudo_id = b.user_pseudo_id
"""

df_behavior = client.query(sql_query_behavior).to_dataframe()


Очистка данных  

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17512 entries, 0 to 17511
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   user_group             17512 non-null  str                
 1   session_start          17470 non-null  datetime64[us, UTC]
 2   purchase_time          610 non-null    datetime64[us, UTC]
 3   transaction_id         610 non-null    str                
 4   category               610 non-null    str                
 5   session_number         610 non-null    Int64              
 6   city                   610 non-null    str                
 7   source_medium          610 non-null    str                
 8   crm_order_id           610 non-null    str                
 9   order_status           610 non-null    str                
 10  created_at             610 non-null    datetime64[us]     
 11  revenue                610 non-null    float64            
 12  p

In [10]:
df_behavior.info()


<class 'pandas.DataFrame'>
RangeIndex: 17494 entries, 0 to 17493
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_group       17494 non-null  str    
 1   user_pseudo_id   17494 non-null  str    
 2   user_item_views  17494 non-null  Int64  
 3   user_carts       17494 non-null  Int64  
 4   user_orders      17494 non-null  Int64  
 5   user_revenue     17494 non-null  float64
dtypes: Int64(3), float64(1), str(2)
memory usage: 1.4 MB


In [11]:
# Проанализировав данные, я заметил, что есть транзакции с отрицательной рентабельностью. Это может быть связано с возвратами товаров или ошибками в данных.
#  Для дальнейшего анализа я решил посмотреть на минимальные значения рентабельности для таких транзакций, чтобы понять, насколько они могут быть убыточными.
negative_profit_df = df[df['profit_margin']<0]

In [12]:
import pandas as pd
from scipy.stats import chisquare

# Подсчет количества уникальных пользователей по группам
observed = df.groupby('user_group')['user_pseudo_id'].nunique().values
expected = [observed.sum() / 2, observed.sum() / 2] # если дизайн был 50/50

# Проведение теста хи-квадрат для проверки SRM (Sample Ratio Mismatch) - проверяем, 
# что распределение пользователей по группам соответствует ожиданиям и 
# что можно принимать результат распределения и рандомизации и на основе єтого делать дальнейший анализ.
# Если p-value меньше 0.05, то мы отвергаем нулевую гипотезу о том, что распределение пользователей по группам соответствует ожиданиям, и можно сделать вывод что распределение полбзователей в тесте было неравномерным.
stat, p_value = chisquare(observed, f_exp=expected)

print(f"P-value для SRM теста: {p_value:.4f}")
if p_value < 0.05:
    print("Обнаружен SRM. Группы распределены неравномерно.")
else:
    print("SRM не обнаружен. Можно делать анализ.")

P-value для SRM теста: 0.7739
SRM не обнаружен. Можно делать анализ.


Результат проверки показал, что распределение пользователей по группам было равномерным и можно продолжать анализ результатов теста. Нулевая гипотеза в данном случае была утверждением что данные распределены равномерно, так как пвелью значительно больше 0,05 мы не можем откинуть данную нулевую гипотезу и считаем что рандомизация и распределение пользователей на сайте реализовано правильно и без перекосов.

1. Проверка на дубликаты

Запрос 1: Если один пользователь совершил 2 покупки, у него будет 2 строки. Это нормально для анализа заказов, но плохо для анализа «конверсии по пользователям»

Запрос 2: Есть ли повторяющиеся user_pseudo_id? Если да — ошибка в логике агрегации SQL.

In [13]:
# duplicates = df['user_pseudo_id'].duplicated().sum()
# print(f"Количество дубликатов в столбце pseudo_id в таблице срм: {duplicates}")

duplicates_behavior = df_behavior['user_pseudo_id'].duplicated().sum()
print(f"Количество дубликатов в столбце user_pseudo_id в таблице поведения: {duplicates_behavior}")

# Переглянемо тих  у кого з'явились дублікати
# У разі якщо у дублікатів буде виявлено що один і той самий корситувач потрапив до обох групп тесту одночасно то це бужде свідчити що розподіл для таких користувачів спрацював з помилкою і таких користувачів треба прибрати із аналізу.
duplicates_behavior_ids = df_behavior[df_behavior['user_pseudo_id'].duplicated()]['user_pseudo_id']
print(df_behavior[df_behavior['user_pseudo_id'].isin(duplicates_behavior_ids)].sort_values('user_pseudo_id'))

Количество дубликатов в столбце user_pseudo_id в таблице поведения: 46
           user_group         user_pseudo_id  user_item_views  user_carts  \
12613  visible_banner  1076882471.1774775761                0           7   
12834   hidden_banner  1076882471.1774775761                0           7   
8744    hidden_banner  1142490368.1757072433                1           0   
8745   visible_banner  1142490368.1757072433                1           0   
2216    hidden_banner  1234155119.1749812765                6           3   
...               ...                    ...              ...         ...   
9183   visible_banner   851565687.1752442304                1           0   
3356    hidden_banner   919759180.1755870406                1           0   
3357   visible_banner   919759180.1755870406                1           0   
14737   hidden_banner   989873055.1769114906                3           0   
14738  visible_banner   989873055.1769114906                3           0   

    

In [14]:
# Видаляю дублікати з таблиці поведінки, щоб не було проблем з аналізом. 
# Так як в попередньому методі я знайшов дублікати то сказати точно чи це були усі хто потрапив в обидві группи і це сталось через помилку GTM чи це дублікати які просто задвоїлися я не можу. 
# Тому для того щоб не видалити потенціно правильні данні які просто задвоїлися я буду робити інтерсекшн по кожній группі щоб знийти лише тих хто став дублем і при цьому потрапив в обидві группи тесту. 
# Це допоможе зберегти правильні данні і видалити лише ті які стали дублем через помилку GTM.

group_a_ids = set(df_behavior[df_behavior['user_group'] == 'hidden_banner']['user_pseudo_id'])
group_b_ids = set(df_behavior[df_behavior['user_group'] == 'visible_banner']['user_pseudo_id'])

dupl_both_groups = group_a_ids.intersection(group_b_ids)

print(f"Кількість користувачів які потрапили в обидві групи: {len(dupl_both_groups)}")

# Видаляю користувачів які потрапили в обидві групи з таблиці поведінки
df_behavior_clean = df_behavior[df_behavior['user_pseudo_id'].isin(dupl_both_groups) == False]

# Перевірка чи залишились дублі по user_pseudo_id після видалення
duplicates_behavior_clean = df_behavior_clean['user_pseudo_id'].duplicated().sum()
print(f"Кількість дублікатів в стовпці user_pseudo_id після видалення: {duplicates_behavior_clean}")
print(f"Залишилось унікальних користувачів: {df_behavior_clean['user_pseudo_id'].nunique()}")

Кількість користувачів які потрапили в обидві групи: 46
Кількість дублікатів в стовпці user_pseudo_id після видалення: 0
Залишилось унікальних користувачів: 17402


Після такої очисти треба ще раз запустити Хі тест щоб перевірити розподіл користувачів у групах. Хоча Оскільки усі хто потрапив до дублікатів мали по дві групи то видлалились рівномірно із кожної групи.

In [15]:
observed_clean = df_behavior_clean.groupby('user_group')['user_pseudo_id'].nunique().values
expected_clean = [observed_clean.sum() / 2, observed_clean.sum() / 2]

stat, p_value = chisquare(observed_clean, f_exp=expected_clean)

print(f"P-value для SRM тесту: {p_value:.4f}")
if p_value < 0.05:
    print("Є перекос. Групи розподілені нерівномірно.")
else:
    print("Перекосу немає. Можна робити аналіз.")

P-value для SRM тесту: 0.7733
Перекосу немає. Можна робити аналіз.


Так як тестуємий банер знаходиться у карточці товару, а на сайті можна додати товар в корзину та зробити замовлення без переходу у карточку товару то таких користувачів треба також видалити із аналізу, щоб аналізувати тільки тих які піддавались впливу тесту тобто перейшли безпосередньо у карточку товару.

In [16]:
# Сформируем датафрейм тех кто был в тесте и видел товар. Это поможет понять, сколько людей действительно  было воронке.
df_exposed_clean = df_behavior_clean[df_behavior_clean['user_item_views'] > 0]
df_exposed_clean.info()

<class 'pandas.DataFrame'>
Index: 7405 entries, 2 to 17490
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_group       7405 non-null   str    
 1   user_pseudo_id   7405 non-null   str    
 2   user_item_views  7405 non-null   Int64  
 3   user_carts       7405 non-null   Int64  
 4   user_orders      7405 non-null   Int64  
 5   user_revenue     7405 non-null   float64
dtypes: Int64(3), float64(1), str(2)
memory usage: 674.3 KB


Розрахунок конверсії. 
Порахуємо конверсію та поведінку користувачів для загального об'єму, тобтьо для абсолютно усих хто прийшов на сайт. І окремо порахуємо лише для тих хто переглянув карточку товару та потенційно побачив банер.

Для чого це. Я хочу подивитись який на справді результат і як він розмивається на фоні загального. 
В нашому випадку різниця conversion_rate у покупку на усих хто потрапив у группи тесту становить 0.1% (3,3%(ті хто не бічив банер) - 3,2%(ті хто бачив банер)) і можна було б сказати що це похибка.
Для тих же хто переглянув картку товару різниця вже значно більше і складає 0,5% п.п., а це 7.2% відносних  (6,9%(ті хто не бічив банер) - 6,4%(ті хто бачив банер)). 
В такому випадку аналіз по всім користувачам міг би приховати проблему. На самомуж ділі банер впливав значно більше на рішення

In [17]:
# Створю одну функцію яка буде рахувати конверсії для подальшого її використання для обидвох таблиць. 

def calc_conversion(df):
    conversion = df.groupby('user_group').agg(
        total_users = ('user_pseudo_id', 'nunique'),
        total_item_views = ('user_item_views', 'sum'),
        total_carts = ('user_carts', 'sum'),
        total_orders = ('user_orders', 'sum'),
        total_revenue = ('user_revenue', 'sum'),
        converted_users = ('user_orders', lambda x: (x > 0).sum())
    )
    conversion['conversion_rate'] = conversion['converted_users'] / conversion['total_users']
    conversion['add_to_cart_rate'] = conversion['total_carts'] / conversion['total_users']
    conversion['purchase_rate'] = conversion['total_orders'] / conversion['total_users']
    conversion['item_view_to_purchase_rate'] = conversion['total_orders'] / conversion['total_item_views']
    return conversion


conversion_all = calc_conversion(df_behavior_clean)
print("Конверсії для всіх учасників тесту:")
display(conversion_all)

conversion_exposed = calc_conversion(df_exposed_clean)
print("Конверсії для учасників тесту, які бачили товар:")
display(conversion_exposed)
    

Конверсії для всіх учасників тесту:


,total_users,total_item_views,total_carts,total_orders,total_revenue,converted_users,conversion_rate,add_to_cart_rate,purchase_rate,item_view_to_purchase_rate
user_group,,,,,,,,,,
hidden_banner,8682,7821,3455,303,268497.0,295,0.033978,0.39795,0.0349,0.038742
visible_banner,8720,7923,3427,291,239558.0,283,0.032454,0.393005,0.033372,0.036729


Конверсії для учасників тесту, які бачили товар:


,total_users,total_item_views,total_carts,total_orders,total_revenue,converted_users,conversion_rate,add_to_cart_rate,purchase_rate,item_view_to_purchase_rate
user_group,,,,,,,,,,
hidden_banner,3652,7821,2773,260,239590.0,252,0.069003,0.75931,0.071194,0.033244
visible_banner,3753,7923,2790,249,211442.0,241,0.064215,0.743405,0.066347,0.031427


Проведу тест для превірки статистичної значущості різниці в конверсіях між групами. Для цьоговикористаю Z-тест. 
В результаті тесту було виявлено що результат не дав статистичної значущості. Але падіння конверсії на 0.5% вказує що цей банер не працює на сбільшення конверсії.
ПОтрібно аналізувати глибше і дивитись на прибуток.

In [18]:
from statsmodels.stats.proportion import proportions_ztest

# данні для сехменту який бачив товар
count = [252, 241] # кількість користувачів які зробили покупку в кожній групі. hidden_banner і visible_banner
nobs = [3652, 3753] # кількість користувачів які бачили товар в кожній групі

stat, p_value = proportions_ztest(count, nobs)

print(f'Z-статистика: {stat:.4f}')
print(f"P-value для тесту пропорцій: {p_value:.4f}")
if p_value < 0.05:
    print("Різниця в конверсії між групами є статистично значущою.")
else:
    print("Різниця в конверсії між групами не є статистично значущою.")



Z-статистика: 0.8263
P-value для тесту пропорцій: 0.4086
Різниця в конверсії між групами не є статистично значущою.


3. Что делать с теми, кто купил без просмотра карточки?
Если твоя гипотеза: «Баннер в карточке увеличивает конверсию», то покупки из категории — это «шум».

Твой план действий:

Создай флаг is_exposed: df['is_exposed'] = df['user_item_views'] > 0.

Сравни две воронки:

Конверсия всех пользователей (ITT).

Конверсия только is_exposed пользователей.

Если в первом случае (все пользователи) результат «не значим», а во втором (те, кто видел баннер) — есть сдвиг, значит, баннер работает, просто его видит слишком мало людей. Это ценнейший инсайт для бизнеса: «Баннер эффективен, давайте вынесем его на главную или в корзину!».

In [19]:
df

,user_group,session_start,purchase_time,transaction_id,category,session_number,city,source_medium,crm_order_id,order_status,...,profit_margin,shipping_costs,payment_name,delivery_service,buyer_lifetime_orders,customer_name,phone,updated_at,logistics_ratio,user_pseudo_id
0,visible_banner,2026-04-05 03:52:52.570147+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,602205754.1775361172
1,hidden_banner,2026-04-05 13:28:12.804623+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1882214553.1775395693
2,hidden_banner,2026-04-05 09:40:25.585726+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,637604898.1771518064
3,hidden_banner,2026-04-05 17:52:33.121968+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1509213045.1775411554
4,visible_banner,2026-04-05 15:54:57.027672+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1578718851.1775404496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17507,hidden_banner,2026-04-06 16:49:00.828928+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1519544007.1775494141
17508,hidden_banner,2026-04-06 11:56:14.257882+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1337102059.1775476574
17509,visible_banner,2026-04-06 13:35:37.619497+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,917709911.1775482536
17510,visible_banner,2026-04-06 20:26:41.193330+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,267966776.1775507199


Треба привести всі значення в таблиці які відсутні але мають значення не нал до одлного вигляду для подальшої роботи

In [20]:
# 1. Сначала убираем лишние пробелы во всех строковых колонках
# Это критично, так как ' None' != 'None'
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# 2. Создаем расширенный список "мусора"
# Используем регулярные выражения (regex=True), чтобы поймать разные регистры
null_regex = r'^([Nn]one|[Nn]an|[Nn]ull|NULL|<[Nn][Aa]>|NA|NaT|)$'

# 3. Применяем замену с переприсваиванием
df = df.replace(null_regex, np.nan, regex=True)

# 4. Специфическая замена для None (объектов Python)
df = df.fillna(value=np.nan)

# ПРОВЕРКА:
print("Количество реальных Null-значений по колонкам:")
print(df.isna().sum())



Количество реальных Null-значений по колонкам:
user_group                   0
session_start               42
purchase_time            16902
transaction_id           16902
category                 16902
session_number           16902
city                     16952
source_medium            16902
crm_order_id             16902
order_status             16902
created_at               16902
revenue                  16902
profit_margin            16902
shipping_costs           16902
payment_name             16968
delivery_service         16902
buyer_lifetime_orders    16902
customer_name            16902
phone                    16902
updated_at               16902
logistics_ratio          16903
user_pseudo_id               0
dtype: int64


In [21]:
ga4_orders = df_exposed_clean['user_orders'].sum()
print(f"Кількість унікальних транзакцій серед тих хто бачив товар: {ga4_orders}")

crm_orders = df['transaction_id'].nunique()
print(f"Кількість унікальних транзакцій серед тих хто не бачив товар: {crm_orders}")

Кількість унікальних транзакцій серед тих хто бачив товар: 509
Кількість унікальних транзакцій серед тих хто не бачив товар: 602


In [22]:
df

,user_group,session_start,purchase_time,transaction_id,category,session_number,city,source_medium,crm_order_id,order_status,...,profit_margin,shipping_costs,payment_name,delivery_service,buyer_lifetime_orders,customer_name,phone,updated_at,logistics_ratio,user_pseudo_id
0,visible_banner,2026-04-05 03:52:52.570147+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,602205754.1775361172
1,hidden_banner,2026-04-05 13:28:12.804623+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1882214553.1775395693
2,hidden_banner,2026-04-05 09:40:25.585726+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,637604898.1771518064
3,hidden_banner,2026-04-05 17:52:33.121968+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1509213045.1775411554
4,visible_banner,2026-04-05 15:54:57.027672+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1578718851.1775404496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17507,hidden_banner,2026-04-06 16:49:00.828928+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1519544007.1775494141
17508,hidden_banner,2026-04-06 11:56:14.257882+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,1337102059.1775476574
17509,visible_banner,2026-04-06 13:35:37.619497+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,917709911.1775482536
17510,visible_banner,2026-04-06 20:26:41.193330+00:00,NaT,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaT,NaN,267966776.1775507199


Нижче розарахуэмо чи э такі замовлення дані про які є в срм але немає даних у га4 та навпаки чи є такі що не потрапили у га4. 
Попереднього скл запит був написаний таким чином щоб отримати тільки ті замовлення дані про які є і га4  і в срм. Але зробимо перевірку.

In [23]:
# Шукаємо тих хто купив GA4, але не є в CRM
missing_in_crm = df[(df['transaction_id'].notnull()) & (df['crm_order_id'].isnull())]

print(f"Кількість тразакцій GA4, яких немає вCRM: {len(missing_in_crm)}")
print(f"Список проблемних транзакцій: {missing_in_crm['transaction_id'].unique()}")

Кількість тразакцій GA4, яких немає вCRM: 0
Список проблемних транзакцій: <ArrowStringArray>
[]
Length: 0, dtype: str


In [24]:
# Шукаю замовлення які є в срм але відсутні у га4.
missing_in_ga4 = df[(df['crm_order_id'].notnull()) & (df['transaction_id'].isnull())]

print(f"Кількість тразакцій CRM, яких немає в GA4: {len(missing_in_ga4)}")

# Розрахуємо відсоток втрати данних
total_crm_orders = df['crm_order_id'].notnull().sum()
gap_percent = (len(missing_in_ga4) / total_crm_orders) * 100
print(f"Відсоток втрачених даних: {gap_percent:.2f}%")

Кількість тразакцій CRM, яких немає в GA4: 0
Відсоток втрачених даних: 0.00%


In [25]:
purchases_crm = df[df['transaction_id'].notnull()]

duplicate_purchase_crm = purchases_crm[purchases_crm['transaction_id'].duplicated(keep=False)]

print(f"Количество дубликатов в столбце transaction_id в таблице срм:{len(duplicate_purchase_crm)}")
display(duplicate_purchase_crm[['user_group', 'transaction_id', 'crm_order_id', 'user_pseudo_id', 'session_start', 'purchase_time', 'revenue']].sort_values('transaction_id'))

Количество дубликатов в столбце transaction_id в таблице срм:16


,user_group,transaction_id,crm_order_id,user_pseudo_id,session_start,purchase_time,revenue
6515,visible_banner,190422,50970,1723472038.1773413797,2026-04-01 20:45:28.858134+00:00,2026-04-02 23:07:33.033781+00:00,1258.0
11142,hidden_banner,190422,50970,1723472038.1773413797,2026-04-01 20:45:28.858134+00:00,2026-04-02 23:07:33.033781+00:00,1258.0
6516,visible_banner,190423,50971,1723472038.1773413797,2026-04-01 20:45:28.858134+00:00,2026-04-02 23:08:49.379734+00:00,1258.0
11143,hidden_banner,190423,50971,1723472038.1773413797,2026-04-01 20:45:28.858134+00:00,2026-04-02 23:08:49.379734+00:00,1258.0
12828,hidden_banner,190468,51019,1920985413.1773496917,2026-04-03 15:19:00.283348+00:00,2026-04-03 15:26:15.978387+00:00,984.0
14874,visible_banner,190468,51019,1920985413.1773496917,2026-04-03 15:19:00.283348+00:00,2026-04-03 15:26:15.978387+00:00,984.0
12771,hidden_banner,190475,51028,837827564.1768070712,2026-04-03 17:48:25.532931+00:00,2026-04-03 18:02:59.904136+00:00,1169.0
13067,visible_banner,190475,51028,837827564.1768070712,2026-04-03 17:48:25.532931+00:00,2026-04-03 18:02:59.904136+00:00,1169.0
5369,hidden_banner,190489,51043,1997279687.1759259004,2026-04-02 19:42:09.877233+00:00,2026-04-03 19:49:49.928346+00:00,925.0
8080,visible_banner,190489,51043,1997279687.1759259004,2026-04-02 19:42:09.877233+00:00,2026-04-03 19:49:49.928346+00:00,925.0


Чистка данних від дублів у таблиці срм серед тих хто зробив замолення. Так як задублювались замовлення при цьому потрапивши одразу у дві группи тесту то такі дані треба повнстю видалити так як не зрозуміло до якої конкртено группи відносився користувач

In [26]:
group_a_ids_crm = set(df[df['user_group'] == 'hidden_banner']['user_pseudo_id'])
group_b_ids_crm = set(df[df['user_group'] == 'visible_banner']['user_pseudo_id'])

intersection_crm = group_a_ids_crm.intersection(group_b_ids_crm) # шукаю користувачів які потрапили в обидві групи в CRM. Це може статися через помилку GTM, коли користувачеві одночасно присвоюється і hidden_banner і visible_banner. Якщо такі користувачі є, то це буде свідчити про те що розподіл для таких користувачів спрацював з помилкою і таких користувачів треба прибрати із аналізу.

print(f"Кількість користувачів які потрапили в обидві групи в CRM: {len(intersection_crm)}")

df_crm_clean = df[df['user_pseudo_id'].isin(intersection_crm) == False] # видаляю користувачів які потрапили в обидві групи з таблиці срм

print(f"Кількість користувачів після видалення тих хто потрапив в обидві групи: {len(df_crm_clean)}")

Кількість користувачів які потрапили в обидві групи в CRM: 46
Кількість користувачів після видалення тих хто потрапив в обидві групи: 17418


In [27]:
print("Дублі айді юзерів:", df_crm_clean['user_pseudo_id'].duplicated().sum())

Дублі айді юзерів: 16


In [28]:
# Находим ID этих 16 дублей
dup_ids = df_crm_clean[df_crm_clean['user_pseudo_id'].duplicated()]['user_pseudo_id'] # це ті ID користувачів які стали дублем після видалення тих хто потрапив в обидві групи. Це може статися якщо у користувача було кілька транзакцій і після видалення деяких транзакцій він став дублем.

# Выводим данные по этим пользователям
# Сортируем по ID, чтобы строки одного юзера стояли рядом
display(df_crm_clean[df_crm_clean['user_pseudo_id'].isin(dup_ids)][['user_pseudo_id', 'transaction_id', 'order_status', 'revenue']].sort_values(by='user_pseudo_id'))

,user_pseudo_id,transaction_id,order_status,revenue
15020,1092174151.1775452543,190693,canceled,699.0
15021,1092174151.1775452543,190694,ПОЛУЧЕН,1258.0
12808,1277916522.1772718915,190465,ПОЛУЧЕН,1352.0
12809,1277916522.1772718915,190467,ПОЛУЧЕН,517.0
15024,1586432771.1775482010,190717,canceled,490.0
15025,1586432771.1775482010,190718,ПОЛУЧЕН,590.0
5139,1618479865.1729438077,190247,canceled,559.0
5140,1618479865.1729438077,190248,canceled,559.0
5523,179918128.1775154334,190398,ПОЛУЧЕН,1258.0
5524,179918128.1775154334,190406,canceled,699.0


в результате по конкретнім дублям для всех пользователей статусы или отменен либо получен. так что надо написать логику которая будет оставлять один заказ в случае если один заказ біл отменен, удалять оба если оба отменены либо оставлять оба если оба получены

In [29]:
print("Статуси замовлень для дублів корситувачів:")
dupl_user_statuses = set(df_crm_clean[df_crm_clean['user_pseudo_id'].isin(dup_ids)]['order_status'].unique())
print(dupl_user_statuses)
print(df_crm_clean[df_crm_clean['user_pseudo_id'].isin(dup_ids)]['order_status'].value_counts(dropna=False))



Статуси замовлень для дублів корситувачів:
{'canceled', 'ПОЛУЧЕН'}
order_status
ПОЛУЧЕН     18
canceled    14
Name: count, dtype: int64


Обробка замовлень для дублів юзерайді. Логіка обробки у настумному. 
1. Якщо у корстувача декілька замовлень зі статусом отримано то залишаємо усі такі замовлення
2. Якщо у користувача є декілька замовлень в рамках однієї сессії(тобто час старту сесіїї однаковий) і статуси замовлень отримано та відмінено, то вважаємо що відмінені то це невдале замовлення і їх треба видалити а зі статусом отримано залишити
3. Якщо всі замовлення зі статусом відмінено то залишаю лише останній замовлення з таким статсуом всі інші видаляю. Також в рамках однієї сесії.

Формую список айді замовлень на видалення та видаляю їхх із основного датафрейму.

In [30]:
df['purchase_time'] = pd.to_datetime(df['purchase_time']) # переконуюсь що purchase_time в форматі datetime для правильного сортування

ids_to_delete = [] # список для зберігання ID транзакцій, які потрібно видалити

grouped = df.groupby(['user_pseudo_id', 'session_start']) # групую по користувачу і сесії, щоб знайти дублікати в межах однієї сесії

for name, group in grouped:
    # Якщо в групі менше 2 записів, то це не дублікати, пропускаємо
    if len(group) < 2:
        continue
        
    statuses = group['order_status'].unique()
    
   # ПРАВИЛО 1: Якщо всі замовлення в сесії "ПОЛУЧЕН"
    if set(statuses) == {'ПОЛУЧЕН'}:
        # Нічого не робимо, залишаємо всі замовлення цієї сесії,
        continue
        
    # ПРАВИЛО 2: Якщо є замовлення зі статусом "ПОЛУЧЕН" і "canceled"
    elif 'ПОЛУЧЕН' in statuses and 'canceled' in statuses:
        # Знаходимо всі замовлення зі статусом "canceled" і додаємо їх ID в список на видалення
        canceled_ids = group[group['order_status'] == 'canceled']['transaction_id'].tolist()
        ids_to_delete.extend(canceled_ids)
        
    # ПРАВИЛО 3: Якщо всі замовлення в сесії "canceled"
    elif set(statuses) == {'canceled'}:
        # Сортуємо замовлення в групі за часом покупки, щоб залишити найновіший, а інші додати в список на видалення
        # Залишаємо найновіший запис, тому сортуємо за спаданням і беремо всі, крім першого
        sorted_ids = group.sort_values(by='purchase_time', ascending=False)['transaction_id'].tolist()
        # Дадаємо всі ID, крім найновішого, в список на видалення
        ids_to_delete.extend(sorted_ids[1:])

# Прибираємо дублікати з основного датафрейму, залишаючи лише один запис для кожного користувача в межах однієї сесії
ids_to_delete = list(set(ids_to_delete))

print(f"Знайдено транзакцій для видалення: {len(ids_to_delete)}")

# Видяляємо транзакції з основного датафрейму
df_clean = df_crm_clean[df_crm_clean['transaction_id'].isin(ids_to_delete) == False].reset_index(drop=True)

print(f"Рядків в ітоговому датафреймі: {len(df_clean)}")


Знайдено транзакцій для видалення: 13
Рядків в ітоговому датафреймі: 17406


In [31]:
print("Дублі айді юзерів:", df_clean['user_pseudo_id'].duplicated().sum())

Дублі айді юзерів: 4


Як видно залишилось чотири юзера із дублями замовлень, це ті які зробили  декілька успішних замовлення 

In [32]:
print("Статуси замовлень для очищенгих дублів корситувачів:")
dup1_ids = df_clean[df_clean['user_pseudo_id'].duplicated()]['user_pseudo_id']
display(df_clean[df_clean['user_pseudo_id'].isin(dup1_ids)][['user_pseudo_id', 'transaction_id', 'order_status', 'revenue']].sort_values(by='user_pseudo_id'))

Статуси замовлень для очищенгих дублів корситувачів:


,user_pseudo_id,transaction_id,order_status,revenue
12732,1277916522.1772718915,190465,ПОЛУЧЕН,1352.0
12733,1277916522.1772718915,190467,ПОЛУЧЕН,517.0
5547,2046625595.1774790854,190377,ПОЛУЧЕН,413.0
5548,2046625595.1774790854,190547,ПОЛУЧЕН,649.0
1318,470460456.1775377549,190597,ПОЛУЧЕН,495.0
1319,470460456.1775377549,190631,ПОЛУЧЕН,447.0
10487,484536791.1775049560,190280,ПОЛУЧЕН,699.0
10488,484536791.1775049560,190284,ПОЛУЧЕН,1258.0
